<a href="https://colab.research.google.com/github/swethasakthianand/aura-climate-twin/blob/main/Rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain langchain-community langchain-openai chromadb bs4


In [ ]:
!pip install google-genai beautifulsoup4 requests numpy

In [ ]:
import os
from google.colab import userdata
from google import genai

# Fetch key from Colab Secrets
GEMINI_KEY = userdata.get("gemini")

# Initialize official Gemini SDK client
client = genai.Client(api_key=GEMINI_KEY)

In [ ]:
import requests
from bs4 import BeautifulSoup

# URL from the freeCodeCamp screenshot
url = "https://lilianweng.github.io/posts/2023-06-23-agent/"
response = requests.get(url)

# Parse HTML
soup = BeautifulSoup(response.content, "html.parser")

# Target the article body content directly
post_content = soup.find("article") or soup.find(class_="post-content")
raw_text = post_content.get_text(separator=" ", strip=True)

print(f"Total characters loaded: {len(raw_text)}")
print("Sample text:", raw_text[:200])

In [ ]:
def chunk_text(text, chunk_size=1000, overlap=200):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap  # Move forward by chunk_size minus overlap
    return chunks


# Generate chunks
chunks = chunk_text(raw_text, chunk_size=1000, overlap=200)

print(f"Total chunks created: {len(chunks)}")
print("Chunk 0:\n", chunks[0][:150])
print("\nChunk 1 (Notice the overlapping start):\n", chunks[1][:150])

In [ ]:
import numpy as np


# Helper function to get 768-dim vector from Gemini
def get_embedding(text):
    response = client.models.embed_content(
        model="text-embedding-004", contents=text
    )
    return response.embedding.values


# Convert text chunks to vectors (limiting to first 20 chunks for speed during testing)
sample_chunks = chunks[:20]
chunk_vectors = [get_embedding(chunk) for chunk in sample_chunks]

# Convert vector list to NumPy Array
vector_db = np.array(chunk_vectors)

print(f"Vector Database shape: {vector_db.shape}")
# Example output: (20, 768) -> 20 rows (chunks), 768 columns (dimensions per vector)

In [ ]:
import numpy as np


# Helper function to get vectors from Gemini
def get_embedding(text):
    response = client.models.embed_content(
        model="gemini-embedding-001", contents=text  # Updated active model name
    )
    # The new gemini-embedding-001 model outputs vector values directly
    return response.embedding.values


# Convert text chunks to vectors (first 20 chunks)
sample_chunks = chunks[:20]
chunk_vectors = [get_embedding(chunk) for chunk in sample_chunks]

# Convert vector list to NumPy Array
vector_db = np.array(chunk_vectors)

print(f"Vector Database shape: {vector_db.shape}")

In [ ]:
import numpy as np


# Helper function to get 768-dim / 3072-dim vector from Gemini
def get_embedding(text):
    response = client.models.embed_content(
        model="gemini-embedding-001", contents=text
    )
    # Access the first item in the embeddings list
    return response.embeddings[0].values


# Convert text chunks to vectors (first 20 chunks)
sample_chunks = chunks[:20]
chunk_vectors = [get_embedding(chunk) for chunk in sample_chunks]

# Convert vector list to NumPy Array
vector_db = np.array(chunk_vectors)

print(f"Vector Database shape: {vector_db.shape}")

In [ ]:
def cosine_similarity(a, b):
    # Dot product divided by the product of vector magnitudes
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


# User Query
query = "What is task decomposition?"

# 1. Convert user query into vector space
query_vector = np.array(get_embedding(query))

# 2. Measure cosine similarity score against every chunk in vector_db
similarity_scores = [
    cosine_similarity(query_vector, chunk_vec) for chunk_vec in vector_db
]

# 3. Find the index of the highest similarity score
best_chunk_index = np.argmax(similarity_scores)
retrieved_context = sample_chunks[best_chunk_index]

print(f"Query: {query}")
print(f"Best Match Score: {similarity_scores[best_chunk_index]:.4f}")
print(f"Retrieved Chunk Index: {best_chunk_index}")
print("\nRetrieved Content:\n", retrieved_context)

In [ ]:
# Construct prompt manually with injected context
prompt = f"""You are a helpful AI assistant. Answer the user question using ONLY the provided context.

Context:
{retrieved_context}

Question:
{query}
"""

# Call Gemini LLM directly
response = client.models.generate_content(
    model="gemini-2.0-flash", contents=prompt
)

print("--- FINAL RAG RESPONSE ---")
print(response.text)

In [ ]:
# Construct prompt manually with injected context
prompt = f"""You are a helpful AI assistant. Answer the user question using ONLY the provided context.

Context:
{retrieved_context}

Question:
{query}
"""

# Try gemini-2.5-flash or gemini-1.5-flash
response = client.models.generate_content(
    model="gemini-2.0-flash", contents=prompt
)

print("--- FINAL RAG RESPONSE ---")
print(response.text)

In [ ]:
!pip install groq

In [ ]:
import os
from google.colab import userdata
from groq import Groq

# Pull Groq key from Colab secrets
groq_client = Groq(api_key=userdata.get("GROQ_API_KEY"))

# Your existing prompt using the retrieved context from earlier steps
prompt = f"""You are a helpful AI assistant. Answer the user question using ONLY the provided context.

Context:
{retrieved_context}

Question:
{query}
"""

# Call Llama-3 model directly to generate the answer
chat_completion = groq_client.chat.completions.create(
    messages=[{"role": "user", "content": prompt}],
    model="llama-3.1-8b-instant",
)

print("--- FINAL RAG RESPONSE ---")
print(chat_completion.choices[0].message.content)

In [22]:
def quick_ask(query_text):
    print("1. Generating query vector...")
    q_vec = np.array(get_embedding(query_text))

    print("2. Searching vector store...")
    scores = [cosine_similarity(q_vec, vec) for vec in vector_db]
    best_idx = np.argmax(scores)

    print("3. Generating answer with Groq...")
    prompt = f"Context:\n{sample_chunks[best_idx]}\n\nQuestion: {query_text}"

    resp = groq_client.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        model="llama-3.1-8b-instant",
    )

    print("\n--- ANSWER ---")
    print(resp.choices[0].message.content)


# Test with one direct call
quick_ask("What is task decomposition?")

1. Generating query vector...
2. Searching vector store...
3. Generating answer with Groq...

--- ANSWER ---
Task decomposition is a process that involves breaking down a complex task or problem into smaller, more manageable subtasks or steps. In the context of the Tree of Thoughts model (Yao et al. 2023) and other AI systems, task decomposition is essential for creating a tree structure of thoughts, where each branch represents a different possible approach to solving the problem.

In the model, task decomposition can be achieved through three methods:

1. **Using a Large Language Model (LLM) with simple prompting**: By providing a prompt such as "Steps for XYZ.\n1." or "What are the subgoals for achieving XYZ?", the LLM can generate multiple possible thoughts or subtasks for each step.

2. **Using task-specific instructions**: The AI system can use task-specific instructions, such as "Write a story outline." for writing a novel, to guide the task decomposition process.

3. **With hum